### DQN（Deep Q-Network，深度Q网络）


| 维度 | 标准 Q-Learning | DQN（深度 Q 网络） |
| :--- | :--- | :--- |
| **Q 值的存储方式** | **查表法（Lookup Table）**：维护一个 $\text{状态数} \times \text{动作数}$ 的二维数组。 | **函数近似（Function Approximation）**：使用神经网络 $Q(s, a; \theta)$ 的权重 $\theta$ 来存储信息。 |
| **输入与输出** | 输入状态 $s$ 的索引，直接查表输出该行所有动作的 Q 值。 | 输入原始状态数据（如游戏像素 $84 \times 84$），网络输出该状态下的所有动作 Q 值。 |
| **适用场景** | **离散、低维、有限**状态空间（如 10x10 迷宫）。 | **连续、高维、海量**状态空间（如下围棋、玩雅达利游戏）。 |
| **泛化能力** | **毫无泛化能力**：没去过的新状态，Q 值就是 0。 | **极强泛化能力**：网络通过参数学习到了状态的特征，类似状态的 Q 值会被推算出相似值。 |

---

### 二者核心差异

- Q-Learning维护Q表，记录所有的状态动作值Q(s,a)，DQN使用深度神经网络 Q(s,a;θ) 去“拟合” Q值
- Q-Learning 无法处理未见过的新状态，而 DQN 可以通过特征相似性进行“推理”

**神经网络融入 Q-Learning**
- 问题：
Q-Learning产生的数据是连续的时间序列$(s_1,α_1,s_2,r_1,α_2,...)$，前后状态高度相关（例如连续两帧游戏画面几乎一样）
神经网络的训练要求数据独立同分布(i.i.d.)否则梯度下降会剧烈震荡，难以收敛。
- 解决方案：
DQN维护一个经验池，存储智能体每一步的交互记录(s,a,r,s')，训练时随机采样一小批样本进行学习。随机采样彻底打破了时间序列的相关性，使得梯度下降变得稳定。同时，可以重复利用旧数据，极大地提升了样本效率。

**移动目标问题**
Q-Learning 的痛点：
在更新公式$r + \gamma \cdot \max_{a'} Q(s', a')$中计算目标的Q和正在更新的Q是同一个网络，导致目标值在动，即模型在追逐一个移动的靶子，使得训练极不稳定

解决方案：
DQN维护两个结构相同的神经网络：
- 评估网络（Q-Network）：用于选择动作，并每一步更新参数θ
- 目标网络（Target-Network）：用于计算 TD Target 中的maxQ(s',α')，其参数$Θ^-$不参与实时更新
- 目标网络的参数$Θ^-$冻结一段时间（比如每 1000 步），然后把评估网络的参数 θ完整复制过去。
可以让目标值（TD Target）在较长时间内保持稳定，使得评估网络有一个清晰、固定的目标可以追赶，极大地抑制了训练震荡。

**损失函数**
DQN 的训练本质上是一个回归（Regression）任务，这与线性回归在思想上高度一致。
$$
L(\theta) = \mathbb{E}_{(s,a,r,s') \sim \text{Replay Buffer}} \left[ \left( \underbrace{r + \gamma \max_{a'} Q(s', a'; \theta^-)}_{\text{TD Target（暂时固定）}} - \underbrace{Q(s, a; \theta)}_{\text{当前预测值}} \right)^2 \right]
$$

- **预测值（Predictions）**  
  由**评估网络**根据当前参数 $\theta$ 计算得出：  
  $$Q(s, a; \theta)$$

- **目标值（Labels / TD Target）**  
  由**目标网络**根据冻结的参数 $\theta^-$ 计算得出，作为学习的“标准答案”：  
  $$y = r + \gamma \max_{a'} Q(s', a'; \theta^-)$$

- **损失函数（均方误差 MSE）**  
  与线性回归的损失函数形式一致，DQN 通过最小化以下期望误差来更新参数 $\theta$：
  
**DQN 缺点**
- **Q 值过估计（Overestimation）**
    因为max操作天然会放大噪声，DQN 经常会高估动作的价值，使用Double DQN（用评估网络选动作，目标网络评价值，将选和评解耦）
- **无法处理连续动作空间**
    DQN 输出的是每个离散动作的 Q 值（如左、右、上、下）。如果动作是“机器人关节转动 30.5 度”这种连续值，DQN无法输出无穷无尽的Q值
- **经验回放偏差**
    均匀随机采样不能区分“平凡经验”和“关键经验”（例如第一次看到目标）
    

In [3]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque
import matplotlib.pyplot as plt

# 1.神经网络
class DQN(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.ReLU(),
            nn.Linear(64, act_dim)
        )
    def forward(self, x):
        return self.net(x)

# 2.经验回放池
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        # 存储时 next_state 已经是 numpy 数组，不为 None
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states),
                np.array(actions),
                np.array(rewards, dtype=np.float32),
                np.array(next_states),        # 现在所有 next_state 都是数组，无 None
                np.array(dones, dtype=np.float32))
    def __len__(self):
        return len(self.buffer)

# 3.超参数 
env = gym.make('CartPole-v1')
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net = DQN(obs_dim, act_dim).to(device)
target_net = DQN(obs_dim, act_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
buffer = ReplayBuffer(10000)
batch_size = 64
gamma = 0.99
eps = 1.0
eps_min = 0.01
eps_decay = 0.995
target_update = 100  # 每100步硬更新目标网络

episode_rewards = []

# 4.训练循环
for episode in range(300):
    state, _ = env.reset()
    state = state.astype(np.float32)
    total_reward = 0
    for step in range(500):
        # ε-greedy 动作选择
        if random.random() < eps:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                q = policy_net(torch.tensor(state, device=device).unsqueeze(0))
                action = q.argmax().item()
        # 执行动作
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state = next_state.astype(np.float32) if not done else np.zeros_like(state)  # ★ 关键修复：用零向量代替 None
        total_reward += reward
        # 存储经验
        buffer.push(state, action, reward, next_state, done)
        state = next_state

        # 训练
        if len(buffer) >= batch_size:
            s, a, r, ns, d = buffer.sample(batch_size)
            s = torch.tensor(s, device=device)
            a = torch.tensor(a, device=device).unsqueeze(1)
            r = torch.tensor(r, device=device)
            ns = torch.tensor(ns, device=device)
            d = torch.tensor(d, device=device)

            # 当前 Q 值
            q_vals = policy_net(s).gather(1, a).squeeze(1)
            # 目标 Q 值
            with torch.no_grad():
                max_next_q = target_net(ns).max(1)[0]
                target = r + gamma * max_next_q * (1 - d)   # 当 d=1 时，target = r
            loss = nn.MSELoss()(q_vals, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # 目标网络硬更新
        if step % target_update == 0:
            target_net.load_state_dict(policy_net.state_dict())

        if done:
            break

    eps = max(eps_min, eps * eps_decay)
    episode_rewards.append(total_reward)

# 5.绘图 
plt.plot(episode_rewards)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('DQN on CartPole (Simplified)')
plt.show()

ModuleNotFoundError: No module named 'gymnasium'